In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

font_path      = "font-folder/Roboto-Light.ttf"
font_path_bold = "font-folder/Roboto-Bold.ttf"
font_prop       = fm.FontProperties(fname=font_path,      size=16)
font_prop_small = fm.FontProperties(fname=font_path,      size=14)
font_prop_title = fm.FontProperties(fname=font_path_bold, size=18)

In [ ]:
# ============================================================
# MODEL SETTINGS
# ============================================================
tsfm_models = [
    'Chronos-2', 'Lag-Llama', 'Moirai-2',
    'TabPFN-TS', 'TimeGPT', 'TimesFM'
]

model_type_color = {
    'Chronos-2':     '#4ECED4',
    'Moirai-2':      '#967ECD',
    'Lag-Llama':    '#3E977E',
    'TabPFN-TS':    '#CE5858',
    'TimeGPT':      '#1F77B4',
    'TimesFM':      '#F2B134',
    'hub-ensemble': 'teal',
    'hub-baseline': '#807B6D'
}

# ============================================================
# LOAD LOCATIONS MAP
# ============================================================
locations_df = pd.read_csv(
    'https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/auxiliary-data/locations.csv'
)
loc_map = dict(zip(
    locations_df['location'].astype(str).str.zfill(2),
    locations_df['abbreviation']
))
us_label = loc_map.get('US', 'US')

In [ ]:
# ============================================================
# DATA LOADING  — accepts metric as parameter
# ============================================================
def load_cleveland_data(challenge, metric):
    pairwise_file = f'forecast_scores_pairwise_comparison_summary_2024-2025_log_{challenge}.csv'
    df = pd.read_csv(
        f'../model-evaluation/forecast_pairwise_comparison_scores/' + 
        pairwise_file
    )
    df = (
        df[['model', 'location', 'wis_scaled_relative_skill', 'ae_median_scaled_relative_skill']]
        .drop_duplicates(subset=['model', 'location'])
        .reset_index(drop=True)
    )
    hub_models = tsfm_models + [f'{challenge}-baseline', f'{challenge}-ensemble']
    df = df[df['model'].isin(hub_models)].copy()
    df['model_name'] = df['model']
    df['model_name'].replace({
        f'{challenge}-baseline': 'hub-baseline',
        f'{challenge}-ensemble': 'hub-ensemble'
    }, inplace=True)
    df['location_abbr'] = df['location'].astype(str).str.zfill(2).map(loc_map)

    # order by median of the requested metric
    model_order = (
        df.groupby('model_name')[metric]
        .median()
        .sort_values()
        .index.tolist()
    )
    return df, model_order


# load four combinations
df_flu_wis,   order_flu_wis   = load_cleveland_data('FluSight', 'wis_scaled_relative_skill')
df_covid_wis, order_covid_wis = load_cleveland_data('CovidHub', 'wis_scaled_relative_skill')
df_flu_ae,    order_flu_ae    = load_cleveland_data('FluSight', 'ae_median_scaled_relative_skill')
df_covid_ae,  order_covid_ae  = load_cleveland_data('CovidHub', 'ae_median_scaled_relative_skill')

In [ ]:
# ============================================================
# PLOT FUNCTIONS
# ============================================================
def plot_cleveland(ax, df, model_order, title, metric):
    n_models = len(model_order)
    yticks   = np.arange(n_models)

    ax.axvline(x=1, color='#888888', linewidth=0.8, linestyle='--', zorder=1)

    for i, model in enumerate(model_order):
        sub    = df[df['model_name'] == model].dropna(subset=[metric])
        color  = model_type_color.get(model, '#aaaaaa')
        values = sub[metric].values
        locs   = sub['location_abbr'].values

        mask_us    = locs == us_label
        mask_state = ~mask_us

        # state dots
        ax.scatter(
            values[mask_state], np.full(mask_state.sum(), i),
            color=color, s=30, linewidths=0, zorder=10
        )

        # US national: hollow square
        if mask_us.any():
            ax.scatter(
                values[mask_us], [i] * mask_us.sum(),
                marker='s', s=60, facecolors='white',
                edgecolors=color, linewidths=1.2, zorder=16
            )

        # median: diamond
        median_val = np.median(values)
        ax.scatter(
            median_val, i,
            marker='D', s=80, color=color,
            edgecolors='white', linewidths=0.8, zorder=16
        )

    ax.set_yticks(yticks)
    ax.set_yticklabels(model_order, fontproperties=font_prop)
    ax.invert_yaxis()
    ax.set_xlabel('Relative score', fontproperties=font_prop)
    for label in ax.get_xticklabels():
        label.set_fontproperties(font_prop_small)

    ax.xaxis.grid(True, linestyle=':', linewidth=0.4, color='#cccccc', zorder=0)
    ax.set_axisbelow(True)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    ax.set_title(title, fontproperties=font_prop_title, pad=10, loc='left', x=0.02)
    ax.set_ylim(n_models - 0.5, -0.5)


def plot_win_bar_h(ax, df, model_order, metric):
    df_states = df[df['location_abbr'] != us_label].copy()
    best_per_state = (
        df_states
        .groupby('location_abbr')
        .apply(lambda x: x.loc[x[metric].idxmin(), 'model_name'])
    )
    win_counts = best_per_state.value_counts().reindex(model_order, fill_value=0)

    n_models = len(model_order)
    for i, model in enumerate(model_order):
        color = model_type_color.get(model, '#aaaaaa')
        ax.barh(i, win_counts[model], color=color, height=0.5, zorder=2)
        if win_counts[model] > 0:
            ax.text(
                win_counts[model] + 0.15, i, str(int(win_counts[model])),
                va='center', fontproperties=font_prop_small
            )

    ax.invert_yaxis()
    ax.set_ylim(n_models - 0.5, -0.5)
    ax.set_yticks([])
    ax.set_xlabel('N wins', fontproperties=font_prop_small)
    for label in ax.get_xticklabels():
        label.set_fontproperties(font_prop_small)
    ax.xaxis.grid(True, linestyle=':', linewidth=0.4, color='#cccccc', zorder=0)
    ax.set_axisbelow(True)
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)

In [ ]:
# ============================================================
# PANEL FIGURE  2×2 + win bars
# layout columns: [cleveland_A, bar_A, gap, cleveland_B, bar_B]
# layout rows:    [WIS (A,B), AE (C,D)]
# ============================================================
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(18, 14), dpi=300)

gs = GridSpec(
    2, 4,
    width_ratios=[4, 1, 4, 1],
    height_ratios=[1, 1],
    wspace=0.05,
    hspace=0.25,
    left=0.06, right=0.97,
    top=0.94, bottom=0.08
)

# row 0: WIS
ax_A     = fig.add_subplot(gs[0, 0])
ax_A_bar = fig.add_subplot(gs[0, 1])
ax_B     = fig.add_subplot(gs[0, 2])
ax_B_bar = fig.add_subplot(gs[0, 3])

# row 1: AE
ax_C     = fig.add_subplot(gs[1, 0])
ax_C_bar = fig.add_subplot(gs[1, 1])
ax_D     = fig.add_subplot(gs[1, 2])
ax_D_bar = fig.add_subplot(gs[1, 3])

# ── plot ─────────────────────────────────────────────────
plot_cleveland(ax_A,     df_flu_wis,   order_flu_wis,   'A.\u2003FluSight \u2013 WIS', 'wis_scaled_relative_skill')
plot_win_bar_h(ax_A_bar, df_flu_wis,   order_flu_wis,   'wis_scaled_relative_skill')

plot_cleveland(ax_B,     df_covid_wis, order_covid_wis, 'B.\u2003CovidHub \u2013 WIS', 'wis_scaled_relative_skill')
plot_win_bar_h(ax_B_bar, df_covid_wis, order_covid_wis, 'wis_scaled_relative_skill')

plot_cleveland(ax_C,     df_flu_ae,    order_flu_ae,    'C.\u2003FluSight \u2013 AE',  'ae_median_scaled_relative_skill')
plot_win_bar_h(ax_C_bar, df_flu_ae,    order_flu_ae,    'ae_median_scaled_relative_skill')

plot_cleveland(ax_D,     df_covid_ae,  order_covid_ae,  'D.\u2003CovidHub \u2013 AE',  'ae_median_scaled_relative_skill')
plot_win_bar_h(ax_D_bar, df_covid_ae,  order_covid_ae,  'ae_median_scaled_relative_skill')

# ── shared ylim per tutti i cleveland ────────────────────
all_cleveland = [ax_A, ax_B, ax_C, ax_D]
xmin_all = min(ax.get_xlim()[0] for ax in all_cleveland)
xmax_all = max(ax.get_xlim()[1] for ax in all_cleveland)
for ax in all_cleveland:
    ax.set_xlim(xmin_all, xmax_all)

# shared xlim per tutti i win bar
all_bars = [ax_A_bar, ax_B_bar, ax_C_bar, ax_D_bar]
wmax_all = max(ax.get_xlim()[1] for ax in all_bars)
for ax in all_bars:
    ax.set_xlim(0, wmax_all)
    
# ── shared xlim within each row ──────────────────────────
for ax_l, ax_r in [(ax_A, ax_B), (ax_C, ax_D)]:
    xmin = min(ax_l.get_xlim()[0], ax_r.get_xlim()[0])
    xmax = max(ax_l.get_xlim()[1], ax_r.get_xlim()[1])
    ax_l.set_xlim(xmin, xmax)
    ax_r.set_xlim(xmin, xmax)

# shared xlim for win bars within each row
for ax_l, ax_r in [(ax_A_bar, ax_B_bar), (ax_C_bar, ax_D_bar)]:
    wmax = max(ax_l.get_xlim()[1], ax_r.get_xlim()[1])
    ax_l.set_xlim(0, wmax)
    ax_r.set_xlim(0, wmax)

# ── shift right panels to avoid overlap ──────────────────
for ax in [ax_B, ax_B_bar, ax_D, ax_D_bar]:
    pos = ax.get_position()
    ax.set_position([pos.x0 + 0.05, pos.y0, pos.width, pos.height])

# ── shared legend ─────────────────────────────────────────
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#888888',
           markersize=6, label='US states'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='white',
           markeredgecolor='#888888', markeredgewidth=1.2,
           markersize=6, label='US national'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#888888',
           markersize=7, label='Median'),
    Line2D([0], [0], color='#888888', linewidth=0.8,
           linestyle='--', label='hub-baseline (1.0)'),
]
fig.legend(
    handles=legend_elements,
    loc='lower center',
    ncol=4,
    frameon=False,
    prop=font_prop_small,
    bbox_to_anchor=(0.5, 0.01)
)

plt.savefig(
    '../model-evaluation/figures/plot_locations/cleveland_plot_log_FluSight-CovidHub_2024-2025.svg',
    dpi=300, bbox_inches='tight'
)
plt.savefig('../model-evaluation/figures/plot_locations/cleveland_plot_log_FluSight-CovidHub_2024-2025.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
df_flu_wis

In [ ]:
df_counts = pd.DataFrame()
for model in df_flu_wis.model.unique():
    df_eval_loc_cleaned_tsfm_model = df_flu_wis[df_flu_wis['model'] == model].copy()
    n_best_wis = df_eval_loc_cleaned_tsfm_model[df_eval_loc_cleaned_tsfm_model['wis_scaled_relative_skill'] < 1].shape[0]
    perc_best_wis =  np.round((n_best_wis / len(df_eval_loc_cleaned_tsfm_model['location'].unique()) * 100))
    n_best_ae = df_eval_loc_cleaned_tsfm_model[df_eval_loc_cleaned_tsfm_model['ae_median_scaled_relative_skill'] < 1].shape[0]
    perc_best_ae =  np.round((n_best_ae / len(df_eval_loc_cleaned_tsfm_model['location'].unique()) * 100))
    new_row = pd.DataFrame({'model': [model], 'perc_best_wis': [perc_best_wis], 'perc_best_ae': [perc_best_ae]})
    df_counts = pd.concat([df_counts, new_row], ignore_index=True)
    print('Model: ', model)
    print(df_eval_loc_cleaned_tsfm_model['wis_scaled_relative_skill'].median(), df_eval_loc_cleaned_tsfm_model['ae_median_scaled_relative_skill'].median())

df_counts

In [ ]:
df_counts = pd.DataFrame()
for model in df_covid_wis.model.unique():
    df_eval_loc_cleaned_tsfm_model = df_covid_wis[df_covid_wis['model'] == model].copy()
    n_best_wis = df_eval_loc_cleaned_tsfm_model[df_eval_loc_cleaned_tsfm_model['wis_scaled_relative_skill'] < 1].shape[0]
    perc_best_wis =  np.round((n_best_wis / len(df_eval_loc_cleaned_tsfm_model['location'].unique()) * 100))
    n_best_ae = df_eval_loc_cleaned_tsfm_model[df_eval_loc_cleaned_tsfm_model['ae_median_scaled_relative_skill'] < 1].shape[0]
    perc_best_ae =  np.round((n_best_ae / len(df_eval_loc_cleaned_tsfm_model['location'].unique()) * 100))
    new_row = pd.DataFrame({'model': [model], 'perc_best_wis': [perc_best_wis], 'perc_best_ae': [perc_best_ae]})
    df_counts = pd.concat([df_counts, new_row], ignore_index=True)
    print('Model: ', model)
    print(df_eval_loc_cleaned_tsfm_model['wis_scaled_relative_skill'].median(), df_eval_loc_cleaned_tsfm_model['ae_median_scaled_relative_skill'].median())

df_counts